# WHI Del 1: Förberedelse av Raster (Medelvärden och Standardavvikelse)

Denna notebook utgör det **första steget** i beräkningen av *Wetland Health Index (WHI)*. Det huvudsakliga syftet är att förbereda och aggregera data som sedan matas in i följande WHI-pipelines (Del 2 och 3). 

Koden aggregerar samtliga tillgängliga (och molnfria) satellitbilder för ett givet studieområde, år för år (under sommarmånaderna juni–augusti), och skapar två typer av utdataraster per index:

1. **Medelvärdesraster (Mean)**: Representerar det genomsnittliga pixelvärdet över säsongen. Har vi 8 bilder från sommaren 2020 beräknas medelvärdet av dessa för varje enskild pixel.
2. **Standardavvikelse (Std)**: Beskriver spridningen/variationen av pixelvärdena under säsongen (pixelstabilitet). 
   - Ett **lågt std-värde** indikerar hög stabilitet (pixelns värde är konstant över tid). 
   - Ett **högt std-värde** indikerar att ytan genomgått en dynamisk förändring under perioden.

Dessa sammanställda raster är grundstenarna som sedan vägs samman och klassificeras i det fortsatta WHI-arbetsflödet.

In [ ]:
import os

from variation import build_and_save_variation_rasters
from config import (
    build_analysis_output_dirs,
    build_analysis_satellites,
    print_satellite_setup,
    )

### Steg 1: Importer och Konfiguration
Här laddas nödvändiga funktioner in från de gemensamma konfigurationsskripten (`config.py` och `variation.py`). Vi sätter också vilket **studieområde** (t.ex. "LF") och vilken **sensor** (t.ex. "sentinel2") som ska analyseras. Även utdatamapparna skapas upp per automatik om de inte redan existerar.

In [ ]:
# ============================================
# Konfigurera sökvägar och studieområde
# ============================================

area = "LF"  # OBS: här väljs studieområdet
sensor = "sentinel2"  # "landsat" eller "sentinel2"

satellites = build_analysis_satellites(area, sensor=sensor)

analysis_output_dirs = build_analysis_output_dirs(sensor=sensor)
output_dir_variation = analysis_output_dirs["variation"]
os.makedirs(output_dir_variation, exist_ok=True)

print_satellite_setup(area, satellites, sensor=sensor)

### Steg 2: Bygg Variation-Raster (Mean / Std)
I det slutgiltiga steget definieras de spektrala index (NDWI, NDVI, osv.) och de årtal som vi önskar summera. Funktionen `build_and_save_variation_rasters` anropas, vilken automatiskt:
- Filtrerar bort molniga pixlar (`max_cloud=0`).
- Begränsar observationerna till de valda sommarmånaderna (`month_start=6`, `month_end=8`).
- Räknar ut summerat medelvärde och standardavvikelse och exporterar dem som nya analytiska Rasterfils-lager (.tif) inför nästa del av WHI-bedömningen.

In [ ]:
# =============================================
# Steg 2: Beräkna och spara variation (mean/std) raster
# =============================================

# Definiera vilka index och år som ska bearbetas
indices = ["NDWI", "NDVI","MNDWI", "NDMI", "EVI"]
years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] # OBS: här väljs året för variation-data

# Kör variation-analysen
build_and_save_variation_rasters(
    satellites=satellites,
    area=area,
    sensor=sensor,
    indices=indices,
    years=years,
    month_start=6,
    month_end=8,
    max_cloud=0
)